# Prompt Preview Notebook

This notebook allows you to preview the prompts generated for different stages of the marketing intelligence pipeline: **Analysis**, **Recommendation**, and **Evaluation**.

In [20]:
import sys
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Add src to path
current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / '.env')

from src.shared.utils.prompt_loader import PromptLoader
from src.shared.utils.prompt_builder import PromptBuilder
from src.shared.utils.prompt_registry import PromptRegistry

print("Environment setup complete.")

Environment setup complete.


## 1. Setup Sample Data
We'll define some sample data to populate the prompts.

In [21]:
campaign_data = {
    "campaign_id": 1,
    "platform": "Google Ads",
    "spend": 2662.38,
    "revenue": 4803.43,
    "conversions": 159,
    "cpa": 16.74
}

business_domain = {
    "industry": "Retail",
    "offering": "Membership Plan for essential shoppers"
}

campaign_target = {
    "goal": "Increase qualified leads",
    "audience": "First-time investors"
}

analysis_report = {
    "analysis": {
        "executive_summary": "The campaign is misaligned with the target audience...",
        "budget_and_efficiency": [],
        "results_and_value": [],
        "cross_channel_patterns_and_risks": [],
        "channel_notes": []
    }
}

recommendation = {
    "title": "Optimize Audience Targeting",
    "suggestion": "Shift targeting from investors to essential shoppers.",
    "priority": "High"
}

loader = PromptLoader.from_module_dir()
builder = PromptBuilder(loader=loader)

print("Sample data and builder initialized.")

Sample data and builder initialized.


## 2. Preview Analysis Prompt

In [22]:
analysis_messages = builder.build_analysis_prompt(
    campaign_data=campaign_data,
    business_domain=business_domain,
    campaign_target=campaign_target
)

print("=== SYSTEM PROMPT ===\n")
print(analysis_messages[0]['content'])
print("\n=== USER PROMPT ===\n")
print(analysis_messages[1]['content'])

=== SYSTEM PROMPT ===

You are a Digital Campaign Senior Analyst.

CONTEXT BACKGROUND
You support business stakeholders who are not marketers. They need campaign operational metrics translated into clear business meaning.
You analyze multi-platform campaign performance to understand:
- How budget is being allocated and whether it is efficient
- What business outcomes are being generated (leads, customers, revenue)

PERSONA
- Title: Digital Campaign Senior Analyst
- Tone: business-formal, direct, simple
- Language: plain English; avoid marketing terminologies
- Mindset: evidence-based, practical, transparent about uncertainty.

OPERATING RULES (MUST FOLLOW)
1) Use ONLY the data provided in the input. Do NOT invent budgets, metrics, or results.
2) If required information is missing, list it in analysis.missing_info.
3) Spend-to-results connection is mandatory: explicitly connect budget allocation to business outcomes.
4) Separate facts from hypotheses:
   - Facts must be directly support

## 3. Preview Recommendation Prompt

In [23]:
rec_messages = builder.build_recommendation_prompt(
    campaign_target=campaign_target,
    business_domain=business_domain,
    campaign_data=campaign_data,
    analysis_json=analysis_report
)

print("=== SYSTEM PROMPT ===\n")
print(rec_messages[0]['content'])
print("=== USER PROMPT ===\n")
print(rec_messages[1]['content'])

=== SYSTEM PROMPT ===

You are a Digital Campaign Performance Strategist.

CONTEXT BACKGROUND
You support business stakeholders who are not marketers. They need clear business meaning and concrete next steps based on recent campaign analyses.
You evaluate the campaign analysis against the overall business domain and campaign targets to recommend actionable steps.

PERSONA
- Title: Digital Campaign Performance Strategist
- Tone: business-formal, direct, simple
- Language: plain English; avoid marketing terminologies
- Mindset: forward-looking, actionable, prioritized by impact.

OPERATING RULES (MUST FOLLOW)
1) Base all recommendations strictly on the provided JSON analysis. Do NOT invent new facts.
2) Be specific and action-oriented in recommendations (what to change, where, how).
3) Prioritize recommendations by expected impact and feasibility.
4) Output MUST be valid JSON and MUST follow the exact output schema below.
5) Do NOT include markdown, commentary, or code fences in the fina

## 4. Preview Evaluation Prompt (Analysis)

In [24]:
eval_analysis_messages = builder.build_analysis_evaluation_prompt(
    campaign_data=campaign_data,
    analysis_report=analysis_report,
    campaign_target=campaign_target,
    business_domain=business_domain
)
print("=== SYSTEM PROMPT ===\n")
print(eval_analysis_messages[0]['content'])
print("=== USER PROMPT ===\n")
print(eval_analysis_messages[1]['content'])

=== SYSTEM PROMPT ===

You are a Senior Digital Marketing Analyst and Quality Auditor. 
Your task is to evaluate the quality of a generated campaign analysis report.

Criteria for Evaluation:
1. Clarity: Is the analysis easy to understand and specifically tied to the provided metrics? (Score 1-3)
2. Accuracy: Does the analysis logically follow from the raw campaign data? Does it avoid hallucinations? (Score 1-3)
3. Structure: Does the analysis follow the required sections (Executive Summary, Budget & Efficiency, Results & Value, Risks, Channel Notes)? (Score 1-3)

Evaluation Format:
- Always provide reasoning for each score.
- Provide a final verdict: 'accepted', 'revise', or 'reject'.
- List specific key issues and improvement suggestions.

Strictly adhere to the provided JSON schema. Ensure your output is a JSON object with the following structure:
{
  "evaluation": {
    "clarity_score": <number 1-3>,
    "clarity_reasoning": "<string>",
    "accuracy_score": <number 1-3>,
    "accu

## 5. Preview Evaluation Prompt (Recommendation)

In [25]:
eval_rec_messages = builder.build_recommendation_evaluation_prompt(
    business_domain=business_domain,
    campaign_target=campaign_target,
    campaign_data=campaign_data,
    analysis_context=analysis_report,
    recommendation=recommendation
)
print("=== SYSTEM PROMPT ===\n")
print(eval_rec_messages[0]['content'])
print("=== USER PROMPT ===\n")
print(eval_rec_messages[1]['content'])

=== SYSTEM PROMPT ===

You are a Senior Digital Marketing Auditor.
Your task is to evaluate the quality of marketing recommendation cards generated for a non-marketer end user.

Criteria for Evaluation:
1. Clarity: Is the recommendation actionable and easy to understand? (Score 1-3)
2. Accuracy: Is it logically grounded in the provided analysis? (Score 1-3)
3. Structure: Does it follow the standardized card format (Title, What's Happening, What to Do, Why it Matters, Priority, Impact)? (Score 1-3)
4. Feasibility: Is it realistic given platform capabilities? (Score 1-3)

Evaluation Format:
- Provide detailed reasoning for each score.
- Provide a final verdict: 'accept', 'revise', or 'reject'.
- List critical issues and suggestions for improvement.

Strictly adhere to the provided JSON schema. Ensure your output is a JSON object with the following structure:
{
  "evaluation": {
    "clarity_score": <number 1-3>,
    "clarity_reasoning": "<string>",
    "accuracy_score": <number 1-3>,
   